# Run EXP-03 - BLIP-2 Fusion VQA

Train and evaluate this EXP from the shared mini HDF5 cache copied to local Colab disk.

## 1. Mount Drive and load repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/blip2-fusion-experiment-vqa"
GITHUB_USER = "theflyingkhui04"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/blip2-fusion-experiment-vqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

Mounted at /content/drive
Cloning into '/content/blip2-fusion-experiment-vqa'...
remote: Enumerating objects: 339, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 339 (delta 46), reused 62 (delta 40), pack-reused 257 (from 1)
Receiving objects: 100% (339/339), 6.27 MiB | 14.48 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/blip2-fusion-experiment-vqa
982f354 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #28 from theflyingkhui04/feat/cdk/template-run
769bed6 (origin/feat/cdk/template-run) Add early-stopping config and mini-cache support
3d34370 Merge pull request #27 from theflyingkhui04/feat/cdk/template-run


## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt -q

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

import wandb
wandb.login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 116.8 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM GB: 15.6


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: trankhanhnhat2k4 (trankhanhnhat2k4-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Choose run

In [ ]:
EXP_ID = "03"
RUN_NUMBER = "1"
YOUR_NAME = "Nhat"
DATA_ROOT = "/content/drive/MyDrive/blip2_project"

CONFIG_FILE = f"configs/exp{EXP_ID}.yaml"
RUN_NAME = f"exp{EXP_ID}_lan{RUN_NUMBER}_{YOUR_NAME}"
CACHE_DIR_NAME = "cache"
DRIVE_CACHE_DIR = f"{DATA_ROOT}/{CACHE_DIR_NAME}"
LOCAL_CACHE_DIR = "/content/blip2_cache"
ANSWER_LIST = f"{DATA_ROOT}/data/ans2idx.json"
OUTPUT_DIR = f"{DATA_ROOT}/checkpoints/{RUN_NAME}"
EVAL_OUTPUT = f"{OUTPUT_DIR}/val_predictions.json"
CHECKPOINT = f"{OUTPUT_DIR}/best_model.pth"

print("Config:", CONFIG_FILE)
print("Run:", RUN_NAME)
print("Drive mini cache:", DRIVE_CACHE_DIR)
print("Local train cache:", LOCAL_CACHE_DIR)
print("Output:", OUTPUT_DIR)

Config: configs/exp03.yaml
Run: exp03_lan1_Nhat
Drive mini cache: /content/drive/MyDrive/blip2_project/cache
Local train cache: /content/blip2_cache
Output: /content/drive/MyDrive/blip2_project/checkpoints/exp03_lan1_Nhat


In [ ]:
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 4. Copy mini cache to local disk

In [ ]:
import os
import shutil
from pathlib import Path
import h5py

drive_mini = {
    "train": Path(DRIVE_CACHE_DIR) / "train_features_mini.h5",
    "val": Path(DRIVE_CACHE_DIR) / "val_features_mini.h5",
}
local_h5 = {
    "train": Path(LOCAL_CACHE_DIR) / "train_features.h5",
    "val": Path(LOCAL_CACHE_DIR) / "val_features.h5",
}
Path(LOCAL_CACHE_DIR).mkdir(parents=True, exist_ok=True)

def sample_cache(path):
    with h5py.File(path, "r") as f:
        key = next(iter(f.keys()))
        return key, f[key].shape, f[key].dtype

for split, source in drive_mini.items():
    if not source.exists():
        raise FileNotFoundError(f"Missing mini cache artifact: {source}")
    print(split, "Drive sample:", sample_cache(source))
    target = local_h5[split]
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)
    print(split, "Local sample:", sample_cache(target))
print("Question subset sizes come from:", CONFIG_FILE)


train Drive sample: ('100014', (257, 1024), dtype('<f2'))
train Local sample: ('100014', (257, 1024), dtype('<f2'))
val Drive sample: ('100008', (257, 1024), dtype('<f2'))
val Local sample: ('100008', (257, 1024), dtype('<f2'))
Question subset sizes come from: configs/exp03.yaml


## 5. Train

In [ ]:
from pathlib import Path

# Ensure OUTPUT_DIR exists and is writable before training
output_path = Path(OUTPUT_DIR)
if not output_path.exists():
    print(f"Recreating missing output directory: {output_path}")
    output_path.mkdir(parents=True, exist_ok=True)

# Verify writability (optional, but good for debugging)
if not os.access(output_path, os.W_OK):
    raise PermissionError(f"Output directory is not writable: {output_path}")

print(f"Output directory confirmed: {output_path}")

Output directory confirmed: /content/drive/MyDrive/blip2_project/checkpoints/exp03_lan1_Nhat


In [ ]:
# Patch file trainer.py để luôn đảm bảo thư mục tồn tại ngay trước lúc lưu model (tránh lỗi Google Drive tự xóa folder trống)
!sed -i 's/torch.save(state, path)/import os; os.makedirs(os.path.dirname(path), exist_ok=True); torch.save(state, path)/g' training/trainer.py
print("Đã patch trainer.py thành công!")

Đã patch trainer.py thành công!


In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}"

2026-06-10 06:51:05 | INFO | __main__ | Random seed: 42
2026-06-10 06:51:05 | INFO | __main__ | Sử dụng device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: trankhanhnhat2k4 (trankhanhnhat2k4-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run aw2h7rv7 (0.0s)
wandb: ⣻ setting up run aw2h7rv7 (0.0s)
wandb: ⣽ setting up run aw2h7rv7 (0.0s)
wandb: ⣾ setting up run aw2h7rv7 (0.0s)
wandb: Tracking run with wandb version 0.27.1
wandb: Run data is saved locally in /content/blip2-fusion-experiment-vqa/wandb/run-20260610_065107-aw2h7rv7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run exp03_lan1_Nhat
wandb: ⭐️ View project at https://wandb.ai/trankhanhnhat2k4-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng/blip2-vqa-experiment
wandb: 🚀 View run at https://wandb.ai/tra

## 6. Evaluate

In [ ]:
!python scripts/evaluate.py \
    --config "{CONFIG_FILE}" \
    --checkpoint "{CHECKPOINT}" \
    --split val \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output "{EVAL_OUTPUT}"

2026-06-10 09:48:42 | INFO | __main__ | Device: cuda
[VQAv2Dataset/val] Cache filter kept 52,669/214,354 samples.
[VQAv2Dataset/val] 40,000 samples | vocab=3,107 | use_cache=True
2026-06-10 09:48:50 | INFO | __main__ | Batches: 1250
2026-06-10 09:48:50 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-10 09:48:50 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-10 09:48:50 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
Loading weights: 100% 199/199 [00:00<00:00, 766.32it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | U

## 7. Resume

In [ ]:
import os
from pathlib import Path

# Tạo lại thư mục lưu checkpoint
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Tạo một file .keep bên trong để Google Drive KHÔNG tự động xóa thư mục trống này nữa
keep_file = output_dir / ".keep"
keep_file.touch()

print(f"Đã tạo thành công thư mục: {output_dir}")
print("Đã thêm file .keep để giữ thư mục an toàn khỏi cơ chế tự xóa của Google Drive!")

Đã tạo thành công thư mục: /content/drive/MyDrive/blip2_project/checkpoints/exp03_lan1_Nhat
Đã thêm file .keep để giữ thư mục an toàn khỏi cơ chế tự xóa của Google Drive!


In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}" \
    --resume auto